In [1]:
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd

from scipy.signal import medfilt

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import io
from scipy.linalg import svd

# Process the coverage of each sample
Reads per position from breseq are corrected for the *lacZ* deletion (VH1000) and the inversion in F, cleaned of the top 1% outliers, and smoothed (200 kb window).
From the smoothed coverage $n(x)$, replication lag (Eq. 3 in Huang et al. 2023) and fork velocity (Eq. 4) are computed:
* lag $= \tau/\ln 2 \cdot \ln(n_{ori}/n(x))$
* fork velocity $= \Delta x / \Delta \mathrm{lag}$, over $\Delta x$ = 100 kb (±50 kb); `fv_censored` masks positions within 50 kb of ori or ter.

The data is then averaged in 5-kb bins and written to `datasets/chromosome_replication/coverage_analysis/intermediate/`.

**Note: the breseq coverage files are too large to include in this repository. When they are absent, this step is skipped and the provided per-sample tables are used.**

In [2]:
# Parameters (including phage in yciI)
phage_size = 46_150
phage_position = 1_310_570

genome_size = 4_641_652 + phage_size # raw genome size of MG1655
ori = 3_925_860 + phage_size # middle of the ori
ter = 1_620_000 + phage_size

# To remove the LacZ gap (deleted in VH1000) (lower coordinates than the phage, no need for correction)
lacz_start = 363_784
lacz_end = 367_161

# Big inversion in F
flip_start = 3_130_743 + phage_size # insH9 near yghO
flip_end = 3_366_154 + phage_size # insH10 near yhcE

# Correct the ori position for the lac deletion
corrected_ori = ori - (lacz_end - lacz_start - 1)
corrected_ter = ter - (lacz_end - lacz_start - 1)
corrected_genome_size = genome_size - (lacz_end - lacz_start - 1)

def periodic_lag(series, lag):
    """
    Lag a series with periodic boundaries.
    Positive lag shifts forward, negative lag shifts backward.
    """
    rolled = np.roll(series.values, lag)
    return pd.Series(rolled, index=series.index)

def get_correct_positions(file, strain, rep):
    '''Read a tab coverage file and apply corrections for the LacZ deletion and the F inversion.'''
                          
    raw = pd.read_csv(file, sep = '\t')

    raw['raw_pos'] = raw.position
    raw['reads'] = raw.unique_top_cov + raw.unique_bot_cov
    raw['redundant_reads'] = raw.redundant_bot_cov + raw.redundant_top_cov

    raw = raw[(raw.raw_pos <= lacz_start) | (raw.raw_pos >= lacz_end)].reset_index(drop = True)

    # Shift the position to make up for the gap
    raw.loc[raw.raw_pos > lacz_start, 'position'] = raw.loc[raw.raw_pos > lacz_start, 'raw_pos'] - (lacz_end - lacz_start - 1)
    raw.rename(columns = {'position': 'shifted_pos'}, inplace=True)

    # Definitive position after all corrections
    raw['pos'] = raw['shifted_pos']
    
    # Flip the F inversion   
    if strain == 'F':
        raw.loc[(raw.raw_pos >= flip_start) & (raw.raw_pos <= flip_end), 'pos'] = raw[(raw.raw_pos >= flip_start) & (raw.raw_pos <= flip_end)].pos.values[::-1]
    
        
    # Add the position relative to origin
    raw['pos_wrt_ori'] = np.where(raw.pos > corrected_ter, raw.pos - corrected_ori, raw.pos - corrected_ori + corrected_genome_size)
    raw['pos_wrt_ter'] = np.where(raw.pos > corrected_ori, raw.pos - corrected_ter - corrected_genome_size, raw.pos - corrected_ter)
    raw = raw.sort_values('pos_wrt_ori').reset_index(drop = True)

    # Remove zeros and redundant regions
    raw.loc[(raw.reads == 0) | (raw.redundant_reads > 0), 'reads'] = np.nan
    
    return raw

def circular_filter(x, filt_window):
    """Apply a rolling-MEAN filter with circular boundaries (not a median, despite the column names below)"""
    
    filt_window = int(filt_window)
    half_window = int(filt_window // 2)
    x = pd.Series(x)

    # Create padded series by adding end to beginning and beginning to end
    padded = pd.concat([
        x[-half_window:], x, x[:half_window]
    ])

    # Now do the rolling mean on padded data and slice back to original size
    filtered = padded.rolling(
        window=filt_window,
        min_periods=1,
        center=True
    ).mean()[half_window:-half_window]
    
    return pd.Series(filtered)

def get_na_mask(x, y):
    return ~(x.isna()) & ~(y.isna())

def clean_signal_tls(expo, stat):
    """Total Least Squares"""
    
    # Remove NAs
    not_na = get_na_mask(expo, stat)
    e = np.array(expo[not_na])
    s = np.array(stat[not_na])
    
    x = s.reshape(-1, 1)
    y = e.reshape(-1, 1)
    X = np.column_stack([x, np.ones_like(x)])
    Z = np.column_stack([X, y])
    
    # Apply SVD
    U, s, Vt = svd(Z, full_matrices=False)
    
    # Get parameters from last right singular vector
    v = Vt[-1, :]
    beta = -v[0] / v[2]  # slope
    
    # Clean signal
    expo_cleaned = expo - beta * stat
    
    # Add the NAs back
    #expo[~(not_na)] = expo_cleaned
    
    return expo_cleaned, beta

In [3]:
# What percentage of points to remove as outliers
outlier_fraction = 0.01
outlier_window_size = int(1e5)

# Final smoothing size to calculate velocities
smoothing_window_size = int(200e3)

# Delay for fork velocity calculation
delay = int(5e4)

In [4]:
# breseq output folders of the marker-frequency samples (not included in this repository: ~150 MB of coverage per sample).
# Each sample folder (B1, B3, F1, F3, R1, R3, W1, W3) is the output of run_breseq_marker_frequency.sh
breseq_output_folder = Path('../../../datasets/chromosome_replication/coverage_analysis/breseq_output/')
intermediate_folder = Path('../../../datasets/chromosome_replication/coverage_analysis/intermediate/')

# Growth rates of the same cultures, measured at harvest time (curve_fitting_MF_epsilon.ipynb)
growth_rates = pd.read_csv('../../../datasets/chromosome_replication/growth_curves/group_params_MF_epsilon.tsv', sep = '\t')


In [5]:
if not breseq_output_folder.exists():
    print('breseq coverage files not found: using the provided per-sample tables in', intermediate_folder)
else:
    experiment_name = 'MF_epsilon'

    expo_files = sorted(breseq_output_folder.glob('*/08_mutation_identification/MG1655_U00096_with_P.coverage.tab'))

    all_dfs = [] # data for plotting
    summary_dfs = [] # summary statistics for each sample

    for file in tqdm(expo_files):
    
        name = file.parent.parent.name # for MFε
        strain = name[0]
        rep = name[1]
    
        # Open the exponential phase data and correct the positions
        raw = get_correct_positions(file, strain, rep)
    
        # Remove noise using knowledge from stationary phase
        # This doesn't really change the results and might introduce artifacts from the stationary data - skip it
        # expo = pd.Series(raw.reads.values)
        # expo_denoised, beta = clean_signal_tls(np.log(expo), log_stat_detrended)
        #raw['reads_denoised'] = np.exp(expo_denoised)
    
        # Smoothen the data
        raw['trend'] = circular_filter(raw['reads'], outlier_window_size)

        # Remove the top outliers
        raw['distance_to_median'] = np.abs(raw['reads'] - raw['trend']) / raw['trend']
        raw['outlier'] = raw['distance_to_median'] > raw['distance_to_median'].quantile(1 - outlier_fraction)
        raw['reads_clean'] = np.where(raw.outlier, np.nan, raw.reads)

        # Final smoothing step
        smoothed = raw[['raw_pos', 'shifted_pos', 'pos', 'pos_wrt_ori', 'pos_wrt_ter', 'reads', 'reads_clean']].reset_index(drop = True)    
        smoothed['reads_smooth'] = circular_filter(smoothed['reads_clean'], smoothing_window_size)
    
        del raw
        
        # Extract replication characteristics
        n_ori = np.mean(smoothed.loc[np.abs(smoothed.pos_wrt_ori) <= 500, 'reads_smooth'])
        n_ter = np.mean(smoothed.loc[np.abs(smoothed.pos_wrt_ter) <= 500, 'reads_smooth'])
    
        tau = growth_rates.loc[(growth_rates.name == name), 'tau'].values[0]
        replication_time = tau / np.log(2) * np.log(n_ori / n_ter)
    
        # Calculate lag times and replication dynamics
        smoothed['lag'] = tau/np.log(2) * np.log(n_ori/smoothed.reads_smooth) # in minutes
    
        d_pos = delay * 2
        d_lag = periodic_lag(smoothed.lag, delay) - periodic_lag(smoothed.lag, -delay)
        smoothed['arm'] = np.where(smoothed.pos_wrt_ori < 0, 1, -1)
        smoothed['fv'] = (d_pos / d_lag) / 60 * smoothed.arm # in bp / s
    
        # Remove the regions too close to ori/ter for velocity to make sense
        smoothed['fv_censored'] = np.where((np.abs(smoothed.pos_wrt_ori) < delay) | (np.abs(smoothed.pos_wrt_ter) < delay), np.nan, smoothed.fv)
    
        # Add to the list of DFs (full data)
        smoothed['name'] = name
        smoothed['strain'] = strain
        smoothed['rep'] = rep
        smoothed['tau'] = tau
        smoothed['n_ori'] = n_ori
        smoothed['n_ter'] = n_ter
        smoothed['rep_time'] = replication_time
    
        # Write the full data for this sample to disk
        #smoothed.to_csv(f'Results/{name}_full_data.tsv', sep = '\t', index = False)
    
        # Make a downsampled version for easier handling
        bin_width = 5000
        arm_length = int(bin_width * np.ceil(corrected_genome_size / (bin_width*2)))
        bins = np.arange(-arm_length, arm_length + bin_width, bin_width)
    
        # Aggregate the counts in 5 kb bins (bins that contain only redundant regions yield NAs)
        smoothed['bin'] = pd.cut(smoothed.pos_wrt_ori, bins, labels = (bins[:-1] + bins[1:]) // 2) # bin centers
        
        compact = smoothed.groupby(['bin', 'name', 'strain', 'rep', 'tau', 'n_ori', 'n_ter', 'rep_time', 'arm'], observed = True).agg({
            'reads': 'mean',
            'reads_smooth': 'mean',
            'pos_wrt_ori': 'mean',
            'pos_wrt_ter': 'mean',
            'raw_pos': 'mean',
            'lag': 'mean',
            'fv': 'mean'
        }).reset_index()
    
        del smoothed
    
        compact['fv_censored'] = np.where((np.abs(compact.pos_wrt_ori) < delay) | (np.abs(compact.pos_wrt_ter) < delay), np.nan, compact.fv)
    
        # Store summary statisics
        summary_stats = pd.DataFrame({'name': [name], 'strain': [strain], 'rep': [rep], 'tau': [tau], 'n_ori': [n_ori], 'n_ter': [n_ter], 'rep_time': [replication_time]})                     
    
        #summary_dfs.append(summary_stats)
        #all_dfs.append(compact)
    
        summary_stats.to_csv(intermediate_folder / f'summary_statistics_{experiment_name}_{name}.tsv', sep = '\t', index = False)
        compact.to_csv(intermediate_folder / f'processed_data_{experiment_name}_{name}.tsv', sep = '\t', index = False)
    
        del compact


breseq coverage files not found: using the provided per-sample tables in ../../../datasets/chromosome_replication/coverage_analysis/intermediate


# Aggregate data

In [6]:
dfs = []
for file in Path('../../../datasets/chromosome_replication/coverage_analysis/intermediate').iterdir():
    if 'processed_data' in file.name:
        print(file)
        dfs.append(pd.read_csv(file, sep='\t'))
        
processed_data = pd.concat(dfs).reset_index(drop = True)
processed_data.to_csv('../../../datasets/chromosome_replication/coverage_analysis/processed_data_MF_epsilon.tsv', sep = '\t', index = False)
#processed_data.to_csv('../../../datasets/chromosome_replication/coverage_analysis/processed_data_E24_beta.tsv', sep = '\t', index = False)

../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_F1.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_F3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_R3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_R1.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_W3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_B3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_B1.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/processed_data_MF_epsilon_W1.tsv


In [7]:
dfs = []
for file in Path('../../../datasets/chromosome_replication/coverage_analysis/intermediate').iterdir():
    if 'summary_statistics' in file.name:
        print(file)
        dfs.append(pd.read_csv(file, sep='\t'))
        
summary_statistics = pd.concat(dfs).reset_index(drop = True)
summary_statistics.to_csv('../../../datasets/chromosome_replication/coverage_analysis/summary_statistics_MF_epsilon.tsv', sep = '\t', index = False)
#summary_statistics.to_csv('../../../datasets/chromosome_replication/coverage_analysis/summary_statistics_E24_beta.tsv', sep = '\t', index = False)

../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_R3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_R1.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_F1.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_F3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_W3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_B3.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_B1.tsv
../../../datasets/chromosome_replication/coverage_analysis/intermediate/summary_statistics_MF_epsilon_W1.tsv
